In [1]:
# importing libraries
import numpy as np 
import pandas as pd
import uproot 
from glob import glob 

In [2]:
branches = ['fEvent', 'fX', 'fY', 'fZ', 'fEdep',
            'fdEdx', 'Ekin', 'TOF', 'TrackLength',
            'ScatteringAng', 'Momentum']

# cargamos los hits 
def load_hits(pattern):
    files = sorted(glob(pattern))
    dfs = []
    for run_id, path in enumerate(files):
        with uproot.open(path) as f:
            tree = f['Hits']
            data = {b: tree[b].array(library='np') for b in branches}
            df = pd.DataFrame({k: v.tolist() for k, v in data.items()})
            df['run_id'] = run_id
            df['event_uid'] = df['run_id'].astype(str) + '_' + df['fEvent'].astype(str)
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)
    

hits_mu = load_hits('data/muon/output_run*.root')
hits_pi = load_hits('data/pion/output_run*.root')

print(hits_mu.shape)
print(hits_pi.shape)

(26668, 13)
(388083, 13)


In [3]:
def aggregate_events(hits_df, label):
    g = hits_df.groupby('event_uid')

    feats = pd.DataFrame(
        {
            "n_hits": g['fEvent'].count(),
            'n_unique_cells': g[['fX', 'fY', 'fZ']].apply(lambda x: len(x.drop_duplicates())),

            # Deposito de energia
            'edep_sum': g['fEdep'].sum(),
            'edep_max': g['fEdep'].max(),
            'edep_std': g['fEdep'].std().fillna(0),

            # dE/dx
            'dedx_mean': g['fdEdx'].mean(),
            'dedx_max': g['fdEdx'].max(),
            'dedx_std': g['fdEdx'].std().fillna(0),

            # energia cinetica 
            'ekin_first': g['Ekin'].first(),
            'ekin_last': g['Ekin'].last(),
            'ekin_loss': g['Ekin'].first() - g['Ekin'].last(),

            # TOF: tiempo de vuelo
            'tof_first': g['TOF'].first(),
            'tof_last': g['TOF'].last(),
            'tof_range': g['TOF'].max() - g['TOF'].min(),

            # Longitud de trayectoria
            'track_first': g['TrackLength'].first(),
            'track_last': g['TrackLength'].last(),
            'track_mean': g['TrackLength'].mean(),

            # Scattering angle 
            'scat_mean': g['ScatteringAng'].mean(),
            'scat_max': g['ScatteringAng'].max(),
            'scat_std': g['ScatteringAng'].std().fillna(0),

            # geometria transversal
            'radial_spread': g.apply(lambda x: np.sqrt(x['fX'].var() + x['fY'].var())).fillna(0),
            'z_span': g['fZ'].max() - g['fZ'].min(),
    })

    feats['label'] = label
    return feats.reset_index()

events_mu = aggregate_events(hits_mu, label=1)
events_pi = aggregate_events(hits_pi, label=0)

df = pd.concat([events_mu, events_pi], ignore_index=True)
print(df.shape)
print(df['label'].value_counts())





/tmp/ipykernel_184631/1304506705.py:40: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  'radial_spread': g.apply(lambda x: np.sqrt(x['fX'].var() + x['fY'].var())).fillna(0),


(39640, 24)
label
0    37686
1     1954
Name: count, dtype: int64


/tmp/ipykernel_184631/1304506705.py:40: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  'radial_spread': g.apply(lambda x: np.sqrt(x['fX'].var() + x['fY'].var())).fillna(0),


In [4]:
df.head()

,event_uid,n_hits,n_unique_cells,edep_sum,edep_max,edep_std,dedx_mean,dedx_max,dedx_std,ekin_first,...,tof_range,track_first,track_last,track_mean,scat_mean,scat_max,scat_std,radial_spread,z_span,label
0,0_0,3,3,0.011298,0.010642,0.005957,0.000519,0.000570,0.000044,9.959221,...,0.010841,40.972700,60.001116,47.431299,0.002405,0.005716,0.002924,8.164966,0.0,1
1,0_1,23,2,0.014892,0.005200,0.001134,0.005381,0.039648,0.008341,9.937587,...,0.177779,53.601376,0.102756,7.939928,0.292034,1.173107,0.341510,4.869848,0.0,1
2,0_10,1,1,0.008106,0.008106,0.000000,0.000405,0.000405,0.000000,9.954342,...,0.000000,60.001189,60.001189,60.001189,0.005239,0.005239,0.000000,0.000000,0.0,1
3,0_100,5,1,0.016102,0.008229,0.003595,0.036192,0.177818,0.079173,9.956423,...,0.056992,40.621146,0.006062,29.514078,0.038343,0.187716,0.083505,0.000000,0.0,1
4,0_101,24,3,0.015754,0.007015,0.001541,0.004796,0.019874,0.005508,9.938223,...,0.113329,40.399218,0.632421,6.620003,0.279870,2.713721,0.553844,6.967835,0.0,1


In [7]:
# Splitting the data into training and testing sets

from itertools import starmap
from sklearn.model_selection import train_test_split 

FEATURES = [c for c in df.columns if c not in ('event_uid', 'label')]

X = df[FEATURES]
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, 
stratify=y, random_state=42)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(31712, 22) (7928, 22) (31712,) (7928,)


In [10]:
from errno import EALREADY

from pandas.core.common import random_state
from xgboost import XGBClassifier
model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    early_stopping_rounds=20,
    random_state=42
)



In [11]:
from tabnanny import verbose


model.fit(X_train, y_train,
eval_set=[(X_test, y_test)],
verbose=50)

[0]	validation_0-logloss:0.15070
[50]	validation_0-logloss:0.00977
[99]	validation_0-logloss:0.00204


,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,20
,enable_categorical,False
,eval_metric,'logloss'
